# 🏃 Opportunity Dataset — Multi-Model Comparison
### CNN · BiGRU · CNN+BiGRU · CNN+BiLSTM · CNN+BiGRU+Gated Attention
**Human Activity Recognition (Locomotion) | Standard Benchmark Split**

**Dataset:** Opportunity Activity Recognition Challenge  
**Task:** Locomotion Mode Classification  
**Subjects:** S1–S4 · 4 locomotion classes · 133 body-worn sensor channels · 30 Hz  

**Sensors:** Body accelerometers (12 locations) · IMU BACK/RUA/RLA/LUA/LLA/L-SHOE/R-SHOE  

**Models compared:**
1. **CNN Only** — 1D Convolutional feature extractor
2. **BiGRU Only** — Bidirectional GRU temporal model
3. **CNN + BiGRU** — Local features → temporal modelling (no attention)
4. **CNN + BiLSTM** — Local features → LSTM temporal modelling (no attention)
5. **CNN + BiGRU + Gated Attention** ← Full hybrid flagship model

**Benchmark split (standard Opportunity protocol):**
- **Train:** S1-ADL1/2, S2-ADL1/2, S3-ADL1/2, S1/S2/S3-Drill
- **Val:** S1-ADL3, S2-ADL3, S3-ADL3
- **Test:** S1-ADL5, S2-ADL5, S3-ADL5

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Uncomment to install dependencies if not already available
# !pip install scikit-learn matplotlib seaborn tensorflow

import os, zipfile, random, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter, defaultdict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_curve, auc,
    cohen_kappa_score, matthews_corrcoef,
    precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 📂 Step 2 — Extract Dataset

Set `ZIP_PATH` to point to your downloaded `opportunity_activity_recognition.zip`.  
The zip will be extracted into a local `./OpportunityUCIDataset/` directory.

In [ ]:
# ── CONFIGURE THIS PATH ───────────────────────────────────────────────────────
ZIP_PATH    = 'opportunity_activity_recognition.zip'   # adjust if needed
EXTRACT_DIR = './'                                     # extract here
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.isdir(os.path.join(EXTRACT_DIR, 'OpportunityUCIDataset')):
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Extraction complete.')
else:
    print('Dataset directory already exists — skipping extraction.')

DATASET_PATH = os.path.join(EXTRACT_DIR, 'OpportunityUCIDataset', 'dataset')
dat_files    = sorted([f for f in os.listdir(DATASET_PATH) if f.endswith('.dat')])
print(f'Dataset root: {DATASET_PATH}')
print(f'.dat files found ({len(dat_files)}): {dat_files}')

## 🔬 Step 3 — Dataset Constants & Column Definitions

**133 body-worn sensor columns** (indices 1–133 in the raw file, i.e. skipping the timestamp at index 0):  

| Column range (raw file, 0-indexed) | Sensor group |
|---|---|
| 1–36 | Body accelerometers (12 positions × 3 axes) |
| 37–49 | IMU BACK (acc + gyro + mag + quaternion) |
| 50–62 | IMU RUA |
| 63–75 | IMU RLA |
| 76–88 | IMU LUA |
| 89–101 | IMU LLA |
| 102–117 | IMU L-SHOE |
| 118–133 | IMU R-SHOE |
| **243** | **Locomotion label** |

**Locomotion classes (null = 0 excluded):**  
1 Stand · 2 Walk · 4 Sit · 5 Lie  → remapped to 0, 1, 2, 3

In [ ]:
# ── Dataset configuration ───────────────────────────────────────────────────
SAMPLING_HZ = 30
WINDOW_SIZE = 90        # 3 s @ 30 Hz  (standard Opportunity window)
STEP_SIZE   = 45        # 50 % overlap

# Body-worn sensor columns (0-indexed in the .dat file, after the timestamp at col 0)
SENSOR_COLS  = list(range(1, 134))          # 133 body sensor channels
LABEL_COL    = 243                          # Locomotion (0-indexed)
N_CHANNELS   = len(SENSOR_COLS)             # 133

# Raw label values present in the file → class index mapping
# Null class (0) is excluded; labels 1/2/4/5 → indices 0/1/2/3
LOCO_LABELS_RAW = {1: 'Stand', 2: 'Walk', 4: 'Sit', 5: 'Lie'}
RAW_TO_IDX      = {1: 0, 2: 1, 4: 2, 5: 3}
IDX_TO_LABEL    = {v: k for k, v in RAW_TO_IDX.items()}   # 0→'Stand', etc.
ACTIVITY_LABELS = {0: 'Stand', 1: 'Walk', 2: 'Sit', 3: 'Lie'}
N_CLASSES       = len(ACTIVITY_LABELS)      # 4

# Standard benchmark split (Ordóñez et al. 2016 DeepConvLSTM protocol)
TRAIN_FILES = [
    'S1-ADL1.dat', 'S1-ADL2.dat',
    'S2-ADL1.dat', 'S2-ADL2.dat',
    'S3-ADL1.dat', 'S3-ADL2.dat',
    'S1-Drill.dat', 'S2-Drill.dat', 'S3-Drill.dat',
]
VAL_FILES  = ['S1-ADL3.dat', 'S2-ADL3.dat', 'S3-ADL3.dat']
TEST_FILES = ['S1-ADL5.dat', 'S2-ADL5.dat', 'S3-ADL5.dat']

print(f'Window size  : {WINDOW_SIZE} samples = {WINDOW_SIZE/SAMPLING_HZ:.1f} s')
print(f'Step size    : {STEP_SIZE} samples  (50 % overlap)')
print(f'N channels   : {N_CHANNELS}')
print(f'N classes    : {N_CLASSES} ({list(ACTIVITY_LABELS.values())})')
print(f'Train files  : {len(TRAIN_FILES)}  {TRAIN_FILES}')
print(f'Val files    : {len(VAL_FILES)}  {VAL_FILES}')
print(f'Test files   : {len(TEST_FILES)}  {TEST_FILES}')

## 📥 Step 4 — Load Files & Sliding-Window Segmentation

In [ ]:
def load_dat_file(filepath):
    """
    Load a single Opportunity .dat file.
    Returns sensor data (N, 133) and locomotion labels (N,).
    NaN values in sensor columns are left as-is here; imputation is done later.
    """
    df = pd.read_csv(filepath, sep=' ', header=None)
    data   = df.iloc[:, SENSOR_COLS].values.astype(np.float32)   # (N, 133)
    labels = df.iloc[:, LABEL_COL].values                        # (N,)
    # Replace NaN labels with 0 (null class → will be discarded)
    labels = np.nan_to_num(labels, nan=0).astype(int)
    return data, labels


def sliding_window(data, labels, window=WINDOW_SIZE, step=STEP_SIZE):
    """
    Segment a continuous time series into overlapping windows.
    Window label = majority vote (excluding null class).
    Windows where the majority label is the null class (0) are discarded.
    """
    X_wins, y_wins = [], []
    n = len(data)
    for start in range(0, n - window + 1, step):
        end      = start + window
        seg      = data[start:end]                  # (window, n_channels)
        seg_lab  = labels[start:end]
        # Majority vote among valid (non-zero) labels
        valid    = seg_lab[seg_lab != 0]
        if len(valid) == 0:
            continue
        maj_lab  = Counter(valid).most_common(1)[0][0]
        if maj_lab not in RAW_TO_IDX:
            continue
        X_wins.append(seg)
        y_wins.append(RAW_TO_IDX[maj_lab])          # remap to 0-3
    return np.array(X_wins, dtype=np.float32), np.array(y_wins, dtype=int)


def load_file_list(file_list, desc=''):
    """Load and window a list of .dat filenames, return concatenated X and y."""
    all_X, all_y = [], []
    for fname in file_list:
        fpath = os.path.join(DATASET_PATH, fname)
        raw_data, raw_labels = load_dat_file(fpath)
        X_w, y_w = sliding_window(raw_data, raw_labels)
        all_X.append(X_w)
        all_y.append(y_w)
        print(f'  {fname:<20}: raw={len(raw_data):7,} rows → {len(X_w):5,} windows')
    X = np.concatenate(all_X, axis=0) if all_X else np.empty((0, WINDOW_SIZE, N_CHANNELS))
    y = np.concatenate(all_y, axis=0) if all_y else np.empty((0,), dtype=int)
    print(f'  ── {desc} total: {X.shape}')
    return X, y


print('Loading TRAIN files ...')
X_train_raw, y_train = load_file_list(TRAIN_FILES, 'TRAIN')

print('\nLoading VAL files ...')
X_val_raw, y_val = load_file_list(VAL_FILES, 'VAL')

print('\nLoading TEST files ...')
X_test_raw, y_test = load_file_list(TEST_FILES, 'TEST')

print(f'\nClass distribution (train): {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Class distribution (val)  : {dict(zip(*np.unique(y_val,   return_counts=True)))}')
print(f'Class distribution (test) : {dict(zip(*np.unique(y_test,  return_counts=True)))}')

## 🧹 Step 5 — NaN Imputation (Column-Mean, Fit on Train Only)

In [ ]:
# Compute per-channel mean over ALL train windows (ignoring NaN)
# Shape: (N_train_windows, WINDOW_SIZE, N_CHANNELS)
train_flat = X_train_raw.reshape(-1, N_CHANNELS)     # (N*T, C)
col_mean   = np.nanmean(train_flat, axis=0)           # (C,) — one mean per channel

def impute_nan(X, col_mean):
    """Replace NaN in (N, T, C) array with per-channel column means."""
    X_out = X.copy()
    for c in range(X_out.shape[-1]):
        channel = X_out[:, :, c]
        nan_mask = np.isnan(channel)
        channel[nan_mask] = col_mean[c]
        X_out[:, :, c] = channel
    return X_out

X_train_imp = impute_nan(X_train_raw, col_mean)
X_val_imp   = impute_nan(X_val_raw,   col_mean)
X_test_imp  = impute_nan(X_test_raw,  col_mean)

nan_after = np.isnan(X_train_imp).sum() + np.isnan(X_val_imp).sum() + np.isnan(X_test_imp).sum()
print(f'NaN values after imputation: {nan_after}  (should be 0)')

## 📊 Step 6 — Per-Channel Z-Score Normalization (Fit on Train Only)

In [ ]:
train_mean = X_train_imp.mean(axis=(0, 1), keepdims=True)    # (1, 1, 133)
train_std  = X_train_imp.std( axis=(0, 1), keepdims=True) + 1e-8

X_train = (X_train_imp - train_mean) / train_std
X_val   = (X_val_imp   - train_mean) / train_std
X_test  = (X_test_imp  - train_mean) / train_std

# One-hot encode labels
y_train_cat = to_categorical(y_train, N_CLASSES)
y_val_cat   = to_categorical(y_val,   N_CLASSES)
y_test_cat  = to_categorical(y_test,  N_CLASSES)

print('Z-score normalization done (fit on train only — no leakage).')
print(f'X_train : {X_train.shape}  range=[{X_train.min():.2f}, {X_train.max():.2f}]')
print(f'X_val   : {X_val.shape}  range=[{X_val.min():.2f},   {X_val.max():.2f}]')
print(f'X_test  : {X_test.shape}  range=[{X_test.min():.2f},  {X_test.max():.2f}]')
print(f'\ny_train one-hot shape: {y_train_cat.shape}')

## 🔍 Step 7 — Exploratory Data Analysis

In [ ]:
activity_names = [ACTIVITY_LABELS[i] for i in range(N_CLASSES)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.Set2(np.linspace(0, 1, N_CLASSES))

for ax, (y_split, title) in zip(axes, [
    (y_train, 'Train'), (y_val, 'Validation'), (y_test, 'Test')
]):
    counts = [np.sum(y_split == i) for i in range(N_CLASSES)]
    bars = ax.bar(activity_names, counts, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title} Set — Class Distribution', fontsize=13, fontweight='bold')
    ax.set_ylabel('Window count')
    ax.set_ylim(0, max(counts) * 1.15)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(c), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Opportunity Locomotion — Class Distribution Across Splits',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('opportunity_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Signal sample plot (first 6 IMU channels from BACK sensor) ────────────────
SAMPLE_CHANNELS = list(range(36, 42))   # IMU BACK: acc xyz + gyro xyz
SAMPLE_NAMES    = ['BACK_accX', 'BACK_accY', 'BACK_accZ',
                   'BACK_gyroX', 'BACK_gyroY', 'BACK_gyroZ']
sample_idx = 0

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, ch_idx, ch_name in zip(axes.flatten(), SAMPLE_CHANNELS, SAMPLE_NAMES):
    ax.plot(X_train[sample_idx, :, ch_idx], lw=1.5, color='steelblue')
    ax.set_title(ch_name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Timestep'); ax.grid(alpha=0.3)

fig.suptitle(
    f'IMU BACK Channels — Sample Window (Activity: {ACTIVITY_LABELS[y_train[sample_idx]]})',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('opportunity_signal_sample.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nWindows per class (train):')
for i in range(N_CLASSES):
    n = np.sum(y_train == i)
    pct = 100 * n / len(y_train)
    print(f'  {ACTIVITY_LABELS[i]:<10}: {n:5d}  ({pct:.1f}%)')

## 🏗️ Step 8 — Shared Components

### Gated Attention Mechanism
```
H: (batch, T, d)
  score  = tanh(H @ W_h)           ← content relevance
  gate   = sigmoid(H @ W_g)        ← information gate
  alpha  = softmax(score ⊙ gate)   ← gated attention weights
  context= Σ_t(alpha_t × H_t)      ← weighted context vector
```

In [ ]:
class GatedAttention(layers.Layer):
    """
    Gated Attention Mechanism.
    Input : H of shape (batch, T, d)
    Output: context (batch, d), attention weights (batch, T)
    """
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        d = input_shape[-1]
        self.W_h = self.add_weight(shape=(d, self.units), name='W_h',
                                   initializer='glorot_uniform', trainable=True)
        self.W_g = self.add_weight(shape=(d, self.units), name='W_g',
                                   initializer='glorot_uniform', trainable=True)
        self.v   = self.add_weight(shape=(self.units, 1), name='v',
                                   initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, H):
        score   = tf.tanh(tf.matmul(H, self.W_h))       # (batch, T, units)
        gate    = tf.sigmoid(tf.matmul(H, self.W_g))    # (batch, T, units)
        gated   = score * gate
        e       = tf.matmul(gated, self.v)               # (batch, T, 1)
        attn_w  = tf.nn.softmax(e, axis=1)               # (batch, T, 1)
        context = tf.reduce_sum(attn_w * H, axis=1)     # (batch, d)
        return context, tf.squeeze(attn_w, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg


def cnn_block(x, filters_1, filters_2, pool_size=2, dropout=0.2, block_id=1):
    """Shared CNN block: 2× Conv1D → BN → ReLU → MaxPool → Dropout."""
    prefix = f'block{block_id}'
    x = layers.Conv1D(filters_1, 3, padding='same', name=f'{prefix}_conv1')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn1')(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv1D(filters_2, 3, padding='same', name=f'{prefix}_conv2')(x)
    x = layers.BatchNormalization(name=f'{prefix}_bn2')(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=pool_size, name=f'{prefix}_pool')(x)
    x = layers.Dropout(dropout)(x)
    return x


print('GatedAttention layer & helper functions defined.')

## 🔨 Step 9 — Model Builders (5 Architectures)

Input shape: `(90, 133)` — 90 timesteps @ 30 Hz, 133 sensor channels.

In [ ]:
INPUT_SHAPE = (WINDOW_SIZE, N_CHANNELS)    # (90, 133)
DR = 0.4                                   # global dropout rate


# ══════════════════════════════════════════════════════════════════════════════
# 1. CNN Only
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_only(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = cnn_block(inp, 64,  64,  block_id=1)   # 90 → 45
    x   = cnn_block(x,  128, 128,  block_id=2)   # 45 → 22
    x   = cnn_block(x,  256, 256,  block_id=3)   # 22 → 11
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='CNN_Only')


# ══════════════════════════════════════════════════════════════════════════════
# 2. BiGRU Only
# ══════════════════════════════════════════════════════════════════════════════
def build_bigru_only(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = layers.Bidirectional(
              layers.GRU(128, return_sequences=True, dropout=0.2), name='bigru_1')(inp)
    x   = layers.Bidirectional(
              layers.GRU(64,  return_sequences=False, dropout=0.2), name='bigru_2')(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='BiGRU_Only')


# ══════════════════════════════════════════════════════════════════════════════
# 3. CNN + BiGRU  (no attention)
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_bigru(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = cnn_block(inp, 64,  64,  block_id=1)   # 90 → 45
    x   = cnn_block(x,  128, 128,  block_id=2)   # 45 → 22
    x   = layers.Bidirectional(
              layers.GRU(128, return_sequences=True, dropout=0.2), name='bigru_1')(x)
    x   = layers.Bidirectional(
              layers.GRU(64,  return_sequences=False, dropout=0.2), name='bigru_2')(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='CNN_BiGRU')


# ══════════════════════════════════════════════════════════════════════════════
# 4. CNN + BiLSTM  (no attention)
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_bilstm(input_shape=INPUT_SHAPE, n_classes=N_CLASSES, dr=DR):
    inp = layers.Input(shape=input_shape, name='input')
    x   = cnn_block(inp, 64,  64,  block_id=1)
    x   = cnn_block(x,  128, 128,  block_id=2)
    x   = layers.Bidirectional(
              layers.LSTM(128, return_sequences=True, dropout=0.2), name='bilstm_1')(x)
    x   = layers.Bidirectional(
              layers.LSTM(64,  return_sequences=False, dropout=0.2), name='bilstm_2')(x)
    x   = layers.Dropout(dr)(x)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    return Model(inputs=inp, outputs=out, name='CNN_BiLSTM')


# ══════════════════════════════════════════════════════════════════════════════
# 5. CNN + BiGRU + Gated Attention  ← FLAGSHIP
# ══════════════════════════════════════════════════════════════════════════════
def build_cnn_bigru_gatedattn(input_shape=INPUT_SHAPE, n_classes=N_CLASSES,
                               gru_units_1=128, gru_units_2=64, attn_units=64,
                               dr=DR):
    inp = layers.Input(shape=input_shape, name='input')

    # ── CNN Blocks ────────────────────────────────────────────────────────────
    x = cnn_block(inp, 64,  64,  block_id=1)    # 90 → 45 timesteps
    x = cnn_block(x,  128, 128,  block_id=2)    # 45 → 22 timesteps

    # ── Bidirectional GRU (return_sequences=True for attention) ──────────────
    x = layers.Bidirectional(
            layers.GRU(gru_units_1, return_sequences=True, dropout=0.2),
            name='bigru_1')(x)
    x = layers.Bidirectional(
            layers.GRU(gru_units_2, return_sequences=True, dropout=0.2),
            name='bigru_2')(x)

    # ── Gated Attention ───────────────────────────────────────────────────────
    context, _ = GatedAttention(units=attn_units, name='gated_attention')(x)

    # ── Classifier Head ───────────────────────────────────────────────────────
    x   = layers.Dropout(dr)(context)
    x   = layers.Dense(128, name='fc1')(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.Dropout(dr / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    return Model(inputs=inp, outputs=out, name='CNN_BiGRU_GatedAttention')


# Print parameter counts for all models
for builder in [build_cnn_only, build_bigru_only, build_cnn_bigru,
                build_cnn_bilstm, build_cnn_bigru_gatedattn]:
    m = builder()
    print(f'{m.name:<35}  params: {m.count_params():>10,}')
    keras.backend.clear_session()

## ⚙️ Step 10 — Training Configuration & Utility

In [ ]:
EPOCHS     = 80
BATCH_SIZE = 64
LR_INIT    = 1e-3


def make_lr_schedule():
    return keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=LR_INIT,
        first_decay_steps=20,
        t_mul=2.0,
        m_mul=0.85,
    )


def make_callbacks(model_name):
    ckpt_path = f'best_{model_name}.weights.h5'
    return [
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=15,
            restore_best_weights=True, verbose=0,
        ),
        callbacks.ModelCheckpoint(
            ckpt_path, monitor='val_accuracy',
            save_best_only=True, save_weights_only=True, verbose=0,
        ),
    ], ckpt_path


def compile_and_train(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=make_lr_schedule()),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    cb_list, ckpt = make_callbacks(model.name)
    print(f'\n{"="*58}')
    print(f' Training: {model.name}  ({model.count_params():,} params)')
    print(f'{"="*58}')
    history = model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=cb_list,
        verbose=1,
    )
    best_val = max(history.history['val_accuracy'])
    print(f'✅ Done — best val accuracy: {best_val:.4f}')
    return history


def evaluate_model(model, history=None):
    """Return a comprehensive metrics dict for all three splits."""
    def _preds(X):
        return model.predict(X, batch_size=BATCH_SIZE, verbose=0)

    proba_test  = _preds(X_test)
    proba_val   = _preds(X_val)
    proba_train = _preds(X_train)

    pred_test   = np.argmax(proba_test,  axis=1)
    pred_val    = np.argmax(proba_val,   axis=1)
    pred_train  = np.argmax(proba_train, axis=1)

    # Per-class AUC (one-vs-rest)
    y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))
    auc_scores = []
    for i in range(N_CLASSES):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba_test[:, i])
        auc_scores.append(auc(fpr, tpr))

    metrics = {
        'name'        : model.name,
        'params'      : model.count_params(),
        'train_acc'   : accuracy_score(y_train, pred_train),
        'val_acc'     : accuracy_score(y_val,   pred_val),
        'test_acc'    : accuracy_score(y_test,  pred_test),
        'weighted_f1' : f1_score(y_test, pred_test, average='weighted'),
        'macro_f1'    : f1_score(y_test, pred_test, average='macro'),
        'micro_f1'    : f1_score(y_test, pred_test, average='micro'),
        'kappa'       : cohen_kappa_score(y_test, pred_test),
        'mcc'         : matthews_corrcoef(y_test, pred_test),
        'mean_auc'    : float(np.mean(auc_scores)),
        'auc_scores'  : auc_scores,
        'proba_test'  : proba_test,
        'pred_test'   : pred_test,
        'pred_val'    : pred_val,
        'history'     : history,
    }
    return metrics


print('Training utilities ready.')
print(f'  Epochs     : {EPOCHS}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  LR init    : {LR_INIT}')

## 🚀 Step 11 — Train Model 1 — CNN Only

In [ ]:
model_cnn = build_cnn_only()
model_cnn.summary()
hist_model_cnn = compile_and_train(model_cnn)
metrics_model_cnn = evaluate_model(model_cnn, hist_model_cnn)

print(f'\n  Train Acc  : {metrics_model_cnn["train_acc"]:.4f}')
print(f'  Val   Acc  : {metrics_model_cnn["val_acc"]:.4f}')
print(f'  Test  Acc  : {metrics_model_cnn["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_cnn["weighted_f1"]:.4f}')
print(f'  Mean AUC   : {metrics_model_cnn["mean_auc"]:.4f}')

## 🚀 Step 12 — Train Model 2 — BiGRU Only

In [ ]:
model_bigru = build_bigru_only()
model_bigru.summary()
hist_model_bigru = compile_and_train(model_bigru)
metrics_model_bigru = evaluate_model(model_bigru, hist_model_bigru)

print(f'\n  Train Acc  : {metrics_model_bigru["train_acc"]:.4f}')
print(f'  Val   Acc  : {metrics_model_bigru["val_acc"]:.4f}')
print(f'  Test  Acc  : {metrics_model_bigru["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_bigru["weighted_f1"]:.4f}')
print(f'  Mean AUC   : {metrics_model_bigru["mean_auc"]:.4f}')

## 🚀 Step 13 — Train Model 3 — CNN + BiGRU

In [ ]:
model_cnn_bigru = build_cnn_bigru()
model_cnn_bigru.summary()
hist_model_cnn_bigru = compile_and_train(model_cnn_bigru)
metrics_model_cnn_bigru = evaluate_model(model_cnn_bigru, hist_model_cnn_bigru)

print(f'\n  Train Acc  : {metrics_model_cnn_bigru["train_acc"]:.4f}')
print(f'  Val   Acc  : {metrics_model_cnn_bigru["val_acc"]:.4f}')
print(f'  Test  Acc  : {metrics_model_cnn_bigru["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_cnn_bigru["weighted_f1"]:.4f}')
print(f'  Mean AUC   : {metrics_model_cnn_bigru["mean_auc"]:.4f}')

## 🚀 Step 14 — Train Model 4 — CNN + BiLSTM

In [ ]:
model_cnn_bilstm = build_cnn_bilstm()
model_cnn_bilstm.summary()
hist_model_cnn_bilstm = compile_and_train(model_cnn_bilstm)
metrics_model_cnn_bilstm = evaluate_model(model_cnn_bilstm, hist_model_cnn_bilstm)

print(f'\n  Train Acc  : {metrics_model_cnn_bilstm["train_acc"]:.4f}')
print(f'  Val   Acc  : {metrics_model_cnn_bilstm["val_acc"]:.4f}')
print(f'  Test  Acc  : {metrics_model_cnn_bilstm["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_cnn_bilstm["weighted_f1"]:.4f}')
print(f'  Mean AUC   : {metrics_model_cnn_bilstm["mean_auc"]:.4f}')

## 🚀 Step 15 — Train Model 5 — CNN + BiGRU + Gated Attention ⭐

In [ ]:
model_flagship = build_cnn_bigru_gatedattn()
model_flagship.summary()
hist_model_flagship = compile_and_train(model_flagship)
metrics_model_flagship = evaluate_model(model_flagship, hist_model_flagship)

print(f'\n  Train Acc  : {metrics_model_flagship["train_acc"]:.4f}')
print(f'  Val   Acc  : {metrics_model_flagship["val_acc"]:.4f}')
print(f'  Test  Acc  : {metrics_model_flagship["test_acc"]:.4f}')
print(f'  Weighted F1: {metrics_model_flagship["weighted_f1"]:.4f}')
print(f'  Mean AUC   : {metrics_model_flagship["mean_auc"]:.4f}')

## 📊 Step 16 — Collect All Results

In [ ]:
all_results = [
    metrics_model_cnn,
    metrics_model_bigru,
    metrics_model_cnn_bigru,
    metrics_model_cnn_bilstm,
    metrics_model_flagship,
]

print('All models evaluated.')
for r in all_results:
    print(f"  {r['name']:<40} test_acc={r['test_acc']:.4f}  wF1={r['weighted_f1']:.4f}")

## 📈 Step 17 — Training Curves (All Models)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(26, 8))

for col, res in enumerate(all_results):
    hist    = res['history'].history
    epochs_ = range(1, len(hist['accuracy']) + 1)
    best_ep = np.argmax(hist['val_accuracy']) + 1

    ax_acc = axes[0, col]
    ax_los = axes[1, col]

    ax_acc.plot(epochs_, hist['accuracy'],     color='royalblue', lw=1.8, label='Train')
    ax_acc.plot(epochs_, hist['val_accuracy'], color='tomato',    lw=1.8, label='Val')
    ax_acc.axvline(best_ep, color='green', ls='--', alpha=0.7)
    ax_acc.set_title(res['name'].replace('_', '\n'), fontsize=9, fontweight='bold')
    ax_acc.set_ylabel('Accuracy') if col == 0 else None
    ax_acc.legend(fontsize=7); ax_acc.grid(alpha=0.3)
    ax_acc.text(0.98, 0.02, f"best={max(hist['val_accuracy']):.3f}",
                transform=ax_acc.transAxes, ha='right', va='bottom', fontsize=8, color='green')

    ax_los.plot(epochs_, hist['loss'],     color='royalblue', lw=1.8, label='Train')
    ax_los.plot(epochs_, hist['val_loss'], color='tomato',    lw=1.8, label='Val')
    ax_los.axvline(best_ep, color='green', ls='--', alpha=0.7)
    ax_los.set_xlabel('Epoch'); ax_los.grid(alpha=0.3)
    ax_los.set_ylabel('Loss') if col == 0 else None

fig.suptitle('Training History — All 5 Models (Opportunity Locomotion)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('opportunity_training_curves_all.png', dpi=130, bbox_inches='tight')
plt.show()

## 🏆 Step 18 — Model Comparison Table

In [ ]:
print('\n' + '='*105)
print(f'  {"Model":<40} {"Params":>10} {"Train Acc":>10} {"Val Acc":>9} {"Test Acc":>9} {"wF1":>7} {"AUC":>7} {"Kappa":>7} {"MCC":>7}')
print('-'*105)
for r in all_results:
    gap_flag = ' ⚠️' if (r['train_acc'] - r['test_acc']) > 0.07 else ''
    print(f"  {r['name']:<40} {r['params']:>10,} {r['train_acc']:>10.4f} {r['val_acc']:>9.4f} "
          f"{r['test_acc']:>9.4f} {r['weighted_f1']:>7.4f} {r['mean_auc']:>7.4f} "
          f"{r['kappa']:>7.4f} {r['mcc']:>7.4f}{gap_flag}")
print('='*105)

best = max(all_results, key=lambda r: r['test_acc'])
print(f"\n🏆 Best model: {best['name']}  →  Test Acc={best['test_acc']:.4f}  wF1={best['weighted_f1']:.4f}")

## 📊 Step 19 — Visual Comparison Bar Chart

In [ ]:
metrics_to_plot = ['test_acc', 'weighted_f1', 'macro_f1', 'mean_auc', 'kappa', 'mcc']
metric_labels   = ['Test Accuracy', 'Weighted F1', 'Macro F1', 'Mean AUC', "Cohen's Kappa", 'MCC']
model_names_short = ['CNN\nOnly', 'BiGRU\nOnly', 'CNN+\nBiGRU', 'CNN+\nBiLSTM', 'CNN+BiGRU\n+GatedAttn']

x = np.arange(len(all_results))
width = 0.13
colors_bar = ['steelblue', 'mediumseagreen', 'darkorange', 'orchid', 'tomato', 'goldenrod']

fig, ax = plt.subplots(figsize=(16, 6))
for j, (m_key, m_label, c) in enumerate(zip(metrics_to_plot, metric_labels, colors_bar)):
    vals   = [r[m_key] for r in all_results]
    offset = (j - len(metrics_to_plot)/2 + 0.5) * width
    bars   = ax.bar(x + offset, vals, width, label=m_label, color=c, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=6, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(model_names_short, fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Opportunity Locomotion — Model Comparison: All Metrics', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('opportunity_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🟦 Step 20 — Confusion Matrices (All Models)

In [ ]:
def plot_confusion_matrix(y_true, y_pred_labels, model_name, save_path):
    cm   = confusion_matrix(y_true, y_pred_labels)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    labels = [ACTIVITY_LABELS[i] for i in range(N_CLASSES)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, fmt, ttl in zip(
        axes,
        [cm, cm_n],
        ['d', '.2%'],
        ['Raw Counts', 'Normalized (Row %)'],
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=labels, yticklabels=labels,
                    linewidths=0.5, ax=ax, annot_kws={'size': 11})
        ax.set_title(f'{model_name} — {ttl}', fontsize=11, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=9)
        ax.set_xlabel('Predicted Label', fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()


for res in all_results:
    plot_confusion_matrix(
        y_test, res['pred_test'],
        res['name'],
        f"opportunity_cm_{res['name'].replace(' ', '_').replace('+', '')}.png"
    )

## 📉 Step 21 — ROC Curves (One-vs-Rest, All Models)

In [ ]:
y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))

fig, axes = plt.subplots(1, 5, figsize=(28, 5))
class_colors = ['steelblue', 'mediumseagreen', 'tomato', 'goldenrod']

for ax, res in zip(axes, all_results):
    for i in range(N_CLASSES):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], res['proba_test'][:, i])
        ax.plot(fpr, tpr, color=class_colors[i], lw=2.0,
                label=f"{ACTIVITY_LABELS[i]} (AUC={res['auc_scores'][i]:.2f})")
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title(f"{res['name']}\nmAUC={res['mean_auc']:.4f}", fontsize=9, fontweight='bold')
    ax.set_xlabel('FPR', fontsize=8)
    ax.set_ylabel('TPR', fontsize=8)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)

fig.suptitle('ROC Curves — Test Set (All Models, Opportunity Locomotion)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('opportunity_roc_all_models.png', dpi=130, bbox_inches='tight')
plt.show()

## 📊 Step 22 — Per-Class Precision / Recall / F1 (All Models)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(28, 5))
x_cls = np.arange(N_CLASSES)
width = 0.25

for ax, res in zip(axes, all_results):
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, res['pred_test'], average=None, labels=list(range(N_CLASSES)))
    ax.bar(x_cls - width, prec, width, label='Precision', color='steelblue',      alpha=0.85)
    ax.bar(x_cls,         rec,  width, label='Recall',    color='mediumseagreen', alpha=0.85)
    ax.bar(x_cls + width, f1,   width, label='F1',        color='tomato',         alpha=0.85)
    ax.set_title(res['name'], fontsize=8, fontweight='bold')
    ax.set_xticks(x_cls)
    ax.set_xticklabels([ACTIVITY_LABELS[i] for i in range(N_CLASSES)],
                        rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score')
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Per-Class Precision / Recall / F1 — Test Set (Opportunity Locomotion)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('opportunity_perclass_all.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔢 Step 23 — Classification Reports (All Models)

In [ ]:
for res in all_results:
    print('\n' + '='*65)
    print(f' Classification Report — {res["name"]}')
    print('='*65)
    print(classification_report(
        y_test, res['pred_test'],
        target_names=[ACTIVITY_LABELS[i] for i in range(N_CLASSES)],
        digits=4,
    ))

## 👁️ Step 24 — Gated Attention Weights Visualisation (Flagship Model)

In [ ]:
# Build a sub-model that exposes the attention weights
gru_out    = model_flagship.get_layer('bigru_2').output
attn_layer = model_flagship.get_layer('gated_attention')
_, attn_out = attn_layer(gru_out)
attn_model  = Model(inputs=model_flagship.input, outputs=attn_out)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for ax, act_idx in zip(axes.flatten(), range(N_CLASSES)):
    candidates = np.where(y_test == act_idx)[0]
    if len(candidates) == 0:
        ax.set_title(f'{ACTIVITY_LABELS[act_idx]} (no sample)', fontsize=9)
        ax.axis('off')
        continue

    idx    = candidates[0]
    sample = X_test[idx:idx+1]                              # (1, 90, 133)
    attn_w = attn_model.predict(sample, verbose=0)[0]       # (T_reduced,)
    T_red  = len(attn_w)
    x_ticks = np.linspace(0, WINDOW_SIZE, T_red)

    raw_sig = sample[0, :, SENSOR_COLS.index(38)]           # IMU BACK accX channel
    ax2 = ax.twinx()
    ax.plot(raw_sig, color='steelblue', lw=1.5, label='BACK IMU accX')
    ax2.bar(x_ticks, attn_w, width=3.5, color='tomato', alpha=0.6, label='Attention')
    ax2.set_ylabel('Attention weight', color='tomato', fontsize=9)
    ax.set_title(f'{ACTIVITY_LABELS[act_idx]}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Timestep', fontsize=9)
    ax.grid(alpha=0.25)
    ax.set_ylabel('Normalized signal', fontsize=9)

fig.suptitle(
    'Gated Attention Weights per Activity — Flagship Model\n'
    'Red bars = attention weight  |  Blue line = IMU BACK accX',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('opportunity_attention_viz.png', dpi=130, bbox_inches='tight')
plt.show()

## 🏆 Step 25 — Final Summary Dashboard

In [ ]:
print()
print('█'*85)
print('   FINAL RESULTS — OPPORTUNITY HUMAN ACTIVITY RECOGNITION (LOCOMOTION)')
print('   5-Model Comparison: CNN · BiGRU · CNN+BiGRU · CNN+BiLSTM · CNN+BiGRU+GatedAttn')
print('█'*85)
print(f'  Dataset          : Opportunity UCI Activity Recognition')
print(f'  Task             : Locomotion mode (Stand / Walk / Sit / Lie)')
print(f'  Activities       : {N_CLASSES} (null class excluded)')
print(f'  Sensor channels  : {N_CHANNELS} (body-worn sensors, cols 1-133)')
print(f'  Sampling rate    : {SAMPLING_HZ} Hz')
print(f'  Window size      : {WINDOW_SIZE} samples = {WINDOW_SIZE/SAMPLING_HZ:.1f} s')
print(f'  Step size        : {STEP_SIZE} samples (50% overlap)')
print(f'  Train split      : {TRAIN_FILES}')
print(f'  Val split        : {VAL_FILES}')
print(f'  Test split       : {TEST_FILES}')
print(f'  Train windows    : {len(X_train):,}')
print(f'  Val   windows    : {len(X_val):,}')
print(f'  Test  windows    : {len(X_test):,}')
print()
print(f'  {"Model":<40} {"Test Acc":>10} {"wF1":>8} {"mAUC":>8} {"Kappa":>8} {"MCC":>8} {"Params":>12}')
print('  ' + '-'*98)
for r in all_results:
    gap  = r['train_acc'] - r['test_acc']
    flag = ' ✅' if gap < 0.05 else ' ⚠️' if gap < 0.10 else ' ❌'
    print(f"  {r['name']:<40} {r['test_acc']:>10.4f} {r['weighted_f1']:>8.4f} {r['mean_auc']:>8.4f} "
          f"{r['kappa']:>8.4f} {r['mcc']:>8.4f} {r['params']:>12,}{flag}")

print()
best = max(all_results, key=lambda r: r['test_acc'])
print(f'  🏆 BEST MODEL : {best["name"]}')
print(f'     Test Accuracy  : {best["test_acc"]:.4f}')
print(f'     Weighted F1    : {best["weighted_f1"]:.4f}')
print(f'     Mean AUC (OvR) : {best["mean_auc"]:.4f}')
print(f"     Train-Test gap : {best['train_acc'] - best['test_acc']:.4f}")
print('█'*85)

## 💾 Step 26 — Save Models & Results Bundle

In [ ]:
import zipfile as zf

# Save all trained models
model_var_names = ['cnn_only', 'bigru_only', 'cnn_bigru', 'cnn_bilstm', 'cnn_bigru_gatedattn']
for m, var_name in zip(
    [model_cnn, model_bigru, model_cnn_bigru, model_cnn_bilstm, model_flagship],
    model_var_names
):
    fname = f'opportunity_{var_name}.keras'
    m.save(fname)
    print(f'Saved: {fname}')

# Save metrics to JSON
results_json = []
for r in all_results:
    results_json.append({
        'model'       : r['name'],
        'params'      : r['params'],
        'train_acc'   : float(r['train_acc']),
        'val_acc'     : float(r['val_acc']),
        'test_acc'    : float(r['test_acc']),
        'weighted_f1' : float(r['weighted_f1']),
        'macro_f1'    : float(r['macro_f1']),
        'micro_f1'    : float(r['micro_f1']),
        'mean_auc'    : float(r['mean_auc']),
        'kappa'       : float(r['kappa']),
        'mcc'         : float(r['mcc']),
        'per_class_auc': {ACTIVITY_LABELS[i]: float(s)
                          for i, s in enumerate(r['auc_scores'])},
    })

with open('opportunity_all_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print('Saved: opportunity_all_results.json')

# Bundle everything into a ZIP
output_files = [
    'opportunity_all_results.json',
    'opportunity_training_curves_all.png',
    'opportunity_model_comparison.png',
    'opportunity_roc_all_models.png',
    'opportunity_perclass_all.png',
    'opportunity_attention_viz.png',
    'opportunity_class_distribution.png',
    'opportunity_signal_sample.png',
] + [f'opportunity_{n}.keras'  for n in model_var_names] \
  + [f"opportunity_cm_{r['name'].replace(' ', '_').replace('+', '')}.png" for r in all_results]

with zf.ZipFile('Opportunity_results.zip', 'w') as zipout:
    for fname in output_files:
        if os.path.exists(fname):
            zipout.write(fname)
            print(f'  + {fname}')

print('\nAll outputs bundled → Opportunity_results.zip')

---
## 📌 Architecture & Methodology Notes

### Dataset
| Property | Value |
|---|---|
| Source | Opportunity Activity Recognition Challenge (UCI) |
| Subjects | 4 (S1–S4) |
| Task | Locomotion mode recognition |
| Activities | 4 (Stand, Walk, Sit, Lie — null class excluded) |
| Sensors | 12 body accelerometers + IMU BACK/RUA/RLA/LUA/LLA/L-SHOE/R-SHOE |
| Channels | 133 body-worn sensor channels |
| Sampling rate | 30 Hz |
| Window size | 90 samples = 3.0 s |
| Overlap | 50 % (step = 45) |
| Window label | Majority vote of non-null sample labels |
| NaN handling | Column-mean imputation (fit on train split only) |

### Benchmark Split (Standard Protocol)
| Split | Files |
|---|---|
| **Train** | S1-ADL1/2, S2-ADL1/2, S3-ADL1/2, S1/S2/S3-Drill |
| **Validation** | S1-ADL3, S2-ADL3, S3-ADL3 |
| **Test** | S1-ADL5, S2-ADL5, S3-ADL5 |

### Models
| Model | Architecture |
|---|---|
| **CNN Only** | 3× (Conv1D×2 + BN + ReLU + MaxPool + Dropout) → GAP → Dense |
| **BiGRU Only** | BiGRU(128, seq) → BiGRU(64, no-seq) → Dense |
| **CNN + BiGRU** | 2× CNN block → BiGRU(128, seq) → BiGRU(64, no-seq) → Dense |
| **CNN + BiLSTM** | 2× CNN block → BiLSTM(128, seq) → BiLSTM(64, no-seq) → Dense |
| **CNN + BiGRU + Gated Attention** ⭐ | 2× CNN block → BiGRU(128, seq) → BiGRU(64, seq) → GatedAttention(64) → Dense |

### Gated Attention
| Component | Formula |
|---|---|
| Content gate | score = tanh(H @ W_h) |
| Information gate | gate = σ(H @ W_g) |
| Attention weights | α = softmax(score ⊙ gate) |
| Context vector | c = Σ_t(α_t × h_t) |

### Training Protocol
| Setting | Value |
|---|---|
| Optimizer | Adam + Cosine Decay Restarts |
| Initial LR | 1e-3 |
| Batch size | 64 |
| Max epochs | 80 |
| Early stopping | patience=15, monitor val_accuracy |
| Regularization | Dropout (0.4), Batch Normalization |
| Normalization | Per-channel Z-score (fit on train only) |